In [1]:
"""
Locality Sensitive Hashing (LSH) demo on the 20 Newsgroups dataset.

Features:
- Shingling: documents -> set of word shingles -> binary vector (sparse)
- MinHash signatures using prime-mod hash functions
- LSH banding: hash signatures into buckets
- Transform: new text -> shingles -> signature -> band hashes
- Retrieval: given a query, return candidate docs and rank by Jaccard

Notes:
- This script downloads the 20 Newsgroups dataset via scikit-learn on first run.
- Internet access is required only for the dataset download; after that, it uses the local cache.
"""

from __future__ import annotations

import re
import random
import hashlib
from dataclasses import dataclass
from typing import List, Tuple, Dict, Iterable, Set, Optional

import numpy as np
from scipy import sparse
from sklearn.datasets import fetch_20newsgroups
import time

#############################################
# 1) Shingling
#############################################


def tokenize(text: str) -> List[str]:
    """Simple word tokenizer: lowercase + alnum tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())


def word_shingles(tokens: List[str], k: int = 3) -> Set[Tuple[str, ...]]:
    """Return set of k-gram word shingles as tuples for a token list."""
    if k <= 0:
        raise ValueError("k must be positive")
    if len(tokens) < k:
        return set()
    return {tuple(tokens[i : i + k]) for i in range(len(tokens) - k + 1)}


def build_shingle_id_map(docs: List[str], k: int = 3) -> Dict[Tuple[str, ...], int]:
    """Scan corpus and assign an ID to each distinct shingle."""
    vocab: Dict[Tuple[str, ...], int] = {}
    next_id = 0
    for doc in docs:
        tokens = tokenize(doc)
        sh = word_shingles(tokens, k)
        for s in sh:
            if s not in vocab:
                vocab[s] = next_id
                next_id += 1
    return vocab


def doc_to_shingle_ids(
    doc: str, vocab: Dict[Tuple[str, ...], int], k: int = 3
) -> Set[int]:
    """Map a document to the set of shingle IDs present in that document."""
    tokens = tokenize(doc)
    sh = word_shingles(tokens, k)
    return {vocab[s] for s in sh if s in vocab}


def docs_to_binary_matrix(
    docs: List[str], vocab: Dict[Tuple[str, ...], int], k: int = 3
) -> sparse.csr_matrix:
    """
    Return a sparse binary (n_docs x |V|) CSR matrix, where entry (i,j)=1 if shingle j appears in doc i.
    """
    rows = []
    cols = []
    for i, doc in enumerate(docs):
        ids = doc_to_shingle_ids(doc, vocab, k)
        rows.extend([i] * len(ids))
        cols.extend(sorted(ids))
    data = np.ones(len(rows), dtype=np.uint8)
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(docs), len(vocab)))
    return mat


#############################################
# 2) MinHash signatures with prime-mod hashes
#############################################


@dataclass
class MinHashParams:
    num_perm: int = 128  # number of hash functions
    prime: Optional[int] = None  # large prime > max shingle id; will be set in fit
    a: Optional[np.ndarray] = None
    b: Optional[np.ndarray] = None


BIG_PRIME = 1009576397


def make_minhash_params(
    num_perm: int = 128, rng: Optional[random.Random] = None
) -> MinHashParams:
    if rng is None:
        rng = random.Random(42)
    p = BIG_PRIME
    a = np.array([rng.randrange(1, p) for _ in range(num_perm)], dtype=np.int64)
    b = np.array([rng.randrange(0, p) for _ in range(num_perm)], dtype=np.int64)
    return MinHashParams(num_perm=num_perm, prime=p, a=a, b=b)


def minhash_signature(shingle_ids: Set[int], params: MinHashParams) -> np.ndarray:
    """
    Compute a MinHash signature vector (length num_perm) for a set of integers.
    Hash functions: h_i(x) = (a_i * x + b_i) mod prime.
    """
    if len(shingle_ids) == 0:
        return np.full(
            params.num_perm, fill_value=np.iinfo(np.int64).max, dtype=np.int64
        )

    sig = np.full(params.num_perm, fill_value=np.iinfo(np.int64).max, dtype=np.int64)
    for x in shingle_ids:
        ax_plus_b = (params.a * x + params.b) % params.prime
        sig = np.minimum(sig, ax_plus_b)
    return sig


def build_signature_matrix(
    binary_matrix: sparse.csr_matrix, params: MinHashParams
) -> np.ndarray:
    """
    Given a binary doc-shingle matrix (n_docs x n_shingles), compute the MinHash signature matrix
    of shape (num_perm x n_docs).
    """
    n_docs, n_shingles = binary_matrix.shape
    sig_mat = np.full(
        (params.num_perm, n_docs), fill_value=np.iinfo(np.int64).max, dtype=np.int64
    )

    # Iterate over each column (shingle) and update signatures for docs that contain it
    bm_csc = binary_matrix.tocsc()
    for shingle_id in range(n_shingles):
        start, end = bm_csc.indptr[shingle_id], bm_csc.indptr[shingle_id + 1]
        doc_indices = bm_csc.indices[start:end]
        if doc_indices.size == 0:
            continue
        hv = (params.a * shingle_id + params.b) % params.prime  # shape (num_perm,)
        # For each doc containing this shingle, take elementwise min
        sig_mat[:, doc_indices] = np.minimum(sig_mat[:, doc_indices], hv[:, None])
    return sig_mat


#############################################
# 3) LSH banding: hash signatures into buckets
#############################################


@dataclass
class LSHParams:
    bands: int

    def __post_init__(self):
        if self.bands <= 0:
            raise ValueError("bands must be positive")


def band_hash(vec: Iterable[int], band_idx: int) -> int:
    """Hash a band (sequence of ints) to an integer bucke"""
    # YOUR CODE GOES HERE
    s = f"{band_idx}:" + ",".join(str(int(x)) for x in vec)
    return int(hashlib.sha1(s.encode("utf-8")).hexdigest(), 16)
    


class LSHIndex:
    """End-to-end LSH index for text with word-shingles + MinHash + banding."""

    def __init__(
        self,
        k_shingle: int = 3,
        num_perm: int = 128,
        bands: int = 32,
        rng: Optional[random.Random] = None,
    ):
        if num_perm % bands != 0:
            raise ValueError("num_perm must be divisible by bands")
        self.k = k_shingle
        self.num_perm = num_perm
        self.bands = bands
        self.rows_per_band = num_perm // bands
        self.rng = rng or random.Random(42)

        # Fitted artifacts
        self.vocab: Dict[Tuple[str, ...], int] = {}
        self.mh_params: Optional[MinHashParams] = None
        self.sig_mat: Optional[np.ndarray] = None  # (num_perm x n_docs)
        self.buckets: List[Dict[int, List[int]]] = (
            []
        )  # per-band dict: bucket_id -> [doc_ids]
        self.docs: List[str] = []

    ################################
    # Indexing - map docs into buckets via LSH
    ################################
    def index(self, documents: List[str]) -> "LSHIndex":
        # 1) Build shingle vocab and binary matrix
        self.docs = documents
        self.vocab = build_shingle_id_map(documents, k=self.k)
        X = docs_to_binary_matrix(documents, self.vocab, k=self.k)

        # 2) MinHash params + signature matrix
        self.mh_params = make_minhash_params(num_perm=self.num_perm, rng=self.rng)
        self.sig_mat = build_signature_matrix(X, self.mh_params)

        # 3) Build LSH buckets per band
        self.buckets = [dict() for _ in range(self.bands)]
        
        # YOUR CODE GOES HERE
        n_docs = len(documents)
        for b in range(self.bands):
            start = b * self.rows_per_band
            end = start + self.rows_per_band
            for doc_id in range(n_docs):
                bucket_id = band_hash(self.sig_mat[start:end, doc_id], b)
                self.buckets[b].setdefault(bucket_id, []).append(doc_id)
        
        return self

    ################################
    # Transform / Hash new text
    ################################
    def transform_text_to_signature(self, text: str) -> np.ndarray:
        if not self.mh_params:
            raise RuntimeError("Index not fitted")
        ids = doc_to_shingle_ids(text, self.vocab, k=self.k)
        return minhash_signature(ids, self.mh_params)

    def transform_text_to_band_hashes(self, text: str) -> List[int]:
        sig = self.transform_text_to_signature(text)
        hashes = []
        for b in range(self.bands):
            start = b * self.rows_per_band
            end = start + self.rows_per_band
            hashes.append(band_hash(sig[start:end], b))
        return hashes

    ################################
    # Retrieval
    ################################
    @staticmethod
    def jaccard(a: Set[int], b: Set[int]) -> float:
        if not a and not b:
            return 1.0
        if not a or not b:
            return 0.0
        return len(a & b) / float(len(a | b))

    def query(
        self, text: str, top_k: int = 10, min_candidates: int = 50
    ) -> List[Tuple[int, float]]:
        """
        Return a ranked list of (doc_id, exact_jaccard) for the query text.
        Candidate set is built from LSH buckets that the query falls into.
        """
        if not self.buckets or self.sig_mat is None:
            raise RuntimeError("Index not fitted")

        # Candidate generation via band buckets
        q_hashes = self.transform_text_to_band_hashes(text)
        cand: Set[int] = set()
        for b, h in enumerate(q_hashes):
            cand.update(self.buckets[b].get(h, []))

        # If too few candidates (can happen if bands are many), fall back to union of top buckets
        if len(cand) < min_candidates:
            for bdict in self.buckets:
                # Heuristic: take the largest few buckets
                largest = sorted(
                    bdict.items(), key=lambda kv: len(kv[1]), reverse=True
                )[:5]
                for _, ids in largest:
                    cand.update(ids)
                if len(cand) >= min_candidates:
                    break

        # Rank candidates by exact Jaccard on shingle sets (could also use signature agreement)
        q_ids = doc_to_shingle_ids(text, self.vocab, k=self.k)
        scores: List[Tuple[int, float]] = []
        for d in cand:
            d_ids = doc_to_shingle_ids(self.docs[d], self.vocab, k=self.k)
            scores.append((d, self.jaccard(q_ids, d_ids)))
        scores.sort(key=lambda t: t[1], reverse=True)
        return scores[:top_k]


#############################################
# 4) End-to-end demo
#############################################


def load_20newsgroups(
    subset: str = "train",
    categories: Optional[List[str]] = None,
    sample: Optional[int] = 2000,
    seed: int = 0,
) -> List[str]:
    """Load 20NG texts (optionally subset categories and sample N docs for speed)."""
    data = fetch_20newsgroups(
        subset=subset, categories=categories, remove=("headers", "footers", "quotes")
    )
    texts = data.data
    texts = [text for text in texts if len(text) >= 100]
    if sample is not None and sample < len(texts):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(texts), size=sample, replace=False)
        texts = [texts[i] for i in idx]
    return texts


def build_index_demo():
    # Load a manageable subset to keep runtime modest
    docs = load_20newsgroups(sample=2000)
    index = LSHIndex(k_shingle=3, num_perm=128, bands=32).index(docs)
    return index, docs


def demo_query(index: LSHIndex, query_text, top_k: int = 10):
    results = index.query(query_text, top_k=top_k)
    # print(f"Query doc id: {qid}")
    print(f"Top {top_k} results:")
    for rank, (doc_id, score) in enumerate(results, 1):
        print(f"{rank:2d}. doc={doc_id:5d}  Jaccard={score:.3f}")


if __name__ == "__main__":
    # Example run
    start = time.perf_counter()
    index, docs = build_index_demo()
    end = time.perf_counter()
    print(f"Index built in {end - start:.2f} seconds over {len(docs)} documents.")

    query_text = docs[10][:100]
    start = time.perf_counter()
    demo_query(index, query_text, top_k=10)
    end = time.perf_counter()
    print(f"Query completed in {end - start:.2f} seconds.")


Index built in 5.21 seconds over 2000 documents.
Top 10 results:
 1. doc=   10  Jaccard=0.667
 2. doc= 1280  Jaccard=0.000
 3. doc= 1153  Jaccard=0.000
 4. doc=    0  Jaccard=0.000
 5. doc=  258  Jaccard=0.000
 6. doc=  133  Jaccard=0.000
 7. doc=  267  Jaccard=0.000
 8. doc= 1274  Jaccard=0.000
 9. doc=  654  Jaccard=0.000
10. doc=  529  Jaccard=0.000
Query completed in 0.01 seconds.
